In [1]:
# ============================================================
# LPPLS Detection Script
# - Input : ./merged_var_input.csv
# - Target columns example: SOLVPN, COPPER
# - Uses price LEVEL series, fits LPPLS on log-price
# - Saves:
#   ./lppls_out/lppls_fit_<target>.csv
#   ./lppls_out/lppls_rolling_<target>.csv
#   ./lppls_out/lppls_detected_<target>.csv
#   ./lppls_out/lppls_fit_<target>.png
#   ./lppls_out/lppls_confidence_<target>.png
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.linalg import lstsq

# =========================
# Config
# =========================
INPUT_CSV = "./merged_var_input.csv"
OUT_DIR = "./lppls_out"
os.makedirs(OUT_DIR, exist_ok=True)

DATE_COL = "Date"
TARGET_COLS = ["COPPER", "SOLVPN"]   # 필요시 하나만 남겨도 됨

# Full-sample fit settings
N_INIT_FULL = 80

# Rolling LPPLS settings
WINDOW = 180              # 약 9개월 수준(거래일 기준 대략)
STEP = 5                  # 5거래일 간격
N_INIT_ROLL = 25          # rolling에서는 계산량 절약
MIN_VALID_POINTS = 120

# LPPLS parameter bounds
# m in (0,1), omega usually around [6,13] 많이 사용
BOUNDS = {
    "tc_rel_min": 1.0,      # tc must be after window end by at least 1 time unit
    "tc_rel_max": 120.0,    # tc up to 120 trading days after window end
    "m_min": 0.01,
    "m_max": 0.99,
    "w_min": 6.0,
    "w_max": 13.0,
}

# Bubble filter rules
FILTERS = {
    "m_min": 0.01,
    "m_max": 0.99,
    "w_min": 6.0,
    "w_max": 13.0,
    "tc_min_ahead": 1.0,
    "tc_max_ahead": 120.0,
    "B_negative_for_positive_bubble": True,   # 양의 버블(가속 상승) 기준
    "min_r2": 0.70,
}

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# =========================
# LPPLS core functions
# =========================
def lppls_design_matrix(t, tc, m, w):
    """
    log p(t) = A + B*(tc-t)^m + C1*(tc-t)^m*cos(w*ln(tc-t)) + C2*(tc-t)^m*sin(w*ln(tc-t))
    """
    dt = tc - t
    if np.any(dt <= 0):
        raise ValueError("tc must be greater than all t in the fitting window.")

    f = dt ** m
    g = f * np.cos(w * np.log(dt))
    h = f * np.sin(w * np.log(dt))
    X = np.column_stack([np.ones_like(t), f, g, h])
    return X


def solve_linear_params(t, y_log, tc, m, w):
    """
    Given nonlinear params (tc, m, w), solve linear params (A, B, C1, C2) by OLS.
    """
    X = lppls_design_matrix(t, tc, m, w)
    beta, _, _, _ = lstsq(X, y_log)
    y_hat = X @ beta
    resid = y_log - y_hat
    sse = np.sum(resid ** 2)
    return beta, y_hat, resid, sse


def lppls_objective(theta, t, y_log):
    tc, m, w = theta

    # bounds hard check
    t_end = t.max()
    ahead = tc - t_end
    if not (BOUNDS["tc_rel_min"] <= ahead <= BOUNDS["tc_rel_max"]):
        return 1e12
    if not (BOUNDS["m_min"] <= m <= BOUNDS["m_max"]):
        return 1e12
    if not (BOUNDS["w_min"] <= w <= BOUNDS["w_max"]):
        return 1e12

    try:
        _, _, _, sse = solve_linear_params(t, y_log, tc, m, w)
        if not np.isfinite(sse):
            return 1e12
        return sse
    except Exception:
        return 1e12


def random_init(t):
    """
    Random initialization for (tc, m, w)
    tc is sampled AFTER the last observation in the window.
    """
    t_end = t.max()
    tc = t_end + np.random.uniform(BOUNDS["tc_rel_min"], BOUNDS["tc_rel_max"])
    m = np.random.uniform(BOUNDS["m_min"], BOUNDS["m_max"])
    w = np.random.uniform(BOUNDS["w_min"], BOUNDS["w_max"])
    return np.array([tc, m, w], dtype=float)


def fit_lppls_multistart(t, y_log, n_init=50):
    """
    Multi-start optimization for LPPLS.
    Returns best fit dict.
    """
    best = None

    bounds = [
        (t.max() + BOUNDS["tc_rel_min"], t.max() + BOUNDS["tc_rel_max"]),  # tc
        (BOUNDS["m_min"], BOUNDS["m_max"]),                                # m
        (BOUNDS["w_min"], BOUNDS["w_max"]),                                # w
    ]

    for _ in range(n_init):
        x0 = random_init(t)

        try:
            res = minimize(
                lppls_objective,
                x0=x0,
                args=(t, y_log),
                method="L-BFGS-B",
                bounds=bounds,
                options={"maxiter": 2000}
            )

            if not res.success and not np.isfinite(res.fun):
                continue

            tc, m, w = res.x
            beta, y_hat, resid, sse = solve_linear_params(t, y_log, tc, m, w)
            A, B, C1, C2 = beta

            sst = np.sum((y_log - y_log.mean()) ** 2)
            r2 = 1.0 - sse / sst if sst > 0 else np.nan

            result = {
                "tc": tc,
                "m": m,
                "w": w,
                "A": A,
                "B": B,
                "C1": C1,
                "C2": C2,
                "sse": sse,
                "r2": r2,
                "y_hat": y_hat,
                "resid": resid,
            }

            if (best is None) or (result["sse"] < best["sse"]):
                best = result

        except Exception:
            continue

    return best


def classify_bubble(result, t_end):
    """
    Positive bubble heuristic:
    - 0 < m < 1
    - 6 < w < 13
    - tc near future
    - B < 0  (for faster-than-exponential upward bubble in log-price)
    - decent fit R^2
    """
    if result is None:
        return 0

    tc = result["tc"]
    m = result["m"]
    w = result["w"]
    B = result["B"]
    r2 = result["r2"]

    ahead = tc - t_end

    conds = [
        FILTERS["m_min"] <= m <= FILTERS["m_max"],
        FILTERS["w_min"] <= w <= FILTERS["w_max"],
        FILTERS["tc_min_ahead"] <= ahead <= FILTERS["tc_max_ahead"],
        r2 >= FILTERS["min_r2"],
    ]

    if FILTERS["B_negative_for_positive_bubble"]:
        conds.append(B < 0)

    return int(all(conds))


def fit_one_series(df, target_col):
    """
    Full sample + rolling LPPLS detection for one target column.
    """
    out = df[[DATE_COL, target_col]].copy()
    out = out.dropna().reset_index(drop=True)

    # positive price only
    out = out[out[target_col] > 0].copy().reset_index(drop=True)
    out[DATE_COL] = pd.to_datetime(out[DATE_COL])

    y = out[target_col].values.astype(float)
    y_log = np.log(y)

    # time index
    t = np.arange(len(out), dtype=float)

    # -------------------------
    # Full-sample fit
    # -------------------------
    best_full = fit_lppls_multistart(t, y_log, n_init=N_INIT_FULL)
    if best_full is None:
        raise RuntimeError(f"LPPLS fitting failed for {target_col}")

    fit_df = out.copy()
    fit_df["log_price"] = y_log
    fit_df["log_price_hat"] = best_full["y_hat"]
    fit_df["price_hat"] = np.exp(best_full["y_hat"])
    fit_df.to_csv(os.path.join(OUT_DIR, f"lppls_fit_{target_col}.csv"), index=False, encoding="utf-8-sig")

    # Save parameter summary
    summary_df = pd.DataFrame([{
        "target": target_col,
        "tc_index": best_full["tc"],
        "tc_date_approx": (
            out[DATE_COL].iloc[-1] + pd.Timedelta(days=int(round(best_full["tc"] - t[-1])))
        ),
        "m": best_full["m"],
        "w": best_full["w"],
        "A": best_full["A"],
        "B": best_full["B"],
        "C1": best_full["C1"],
        "C2": best_full["C2"],
        "r2": best_full["r2"],
        "sse": best_full["sse"],
        "is_positive_bubble": classify_bubble(best_full, t[-1]),
    }])
    summary_df.to_csv(os.path.join(OUT_DIR, f"lppls_summary_{target_col}.csv"), index=False, encoding="utf-8-sig")

    # Plot full fit
    plt.figure(figsize=(12, 6))
    plt.plot(out[DATE_COL], y, label="Actual Price")
    plt.plot(out[DATE_COL], fit_df["price_hat"], label="LPPLS Fit")
    plt.title(f"LPPLS Fit - {target_col}")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"lppls_fit_{target_col}.png"), dpi=300)
    plt.close()

    # -------------------------
    # Rolling detection
    # -------------------------
    rolling_rows = []

    n = len(out)
    for end_idx in range(WINDOW, n, STEP):
        start_idx = end_idx - WINDOW
        sub = out.iloc[start_idx:end_idx].copy()

        if len(sub) < MIN_VALID_POINTS:
            continue

        sub_y = sub[target_col].values.astype(float)
        if np.any(sub_y <= 0):
            continue

        sub_log = np.log(sub_y)
        sub_t = np.arange(len(sub), dtype=float)

        best_roll = fit_lppls_multistart(sub_t, sub_log, n_init=N_INIT_ROLL)

        if best_roll is None:
            rolling_rows.append({
                "window_start": sub[DATE_COL].iloc[0],
                "window_end": sub[DATE_COL].iloc[-1],
                "tc_index": np.nan,
                "tc_days_ahead": np.nan,
                "tc_date_approx": pd.NaT,
                "m": np.nan,
                "w": np.nan,
                "A": np.nan,
                "B": np.nan,
                "C1": np.nan,
                "C2": np.nan,
                "r2": np.nan,
                "bubble_flag": 0,
            })
            continue

        tc_days_ahead = best_roll["tc"] - sub_t[-1]
        tc_date_approx = sub[DATE_COL].iloc[-1] + pd.Timedelta(days=int(round(tc_days_ahead)))

        bubble_flag = classify_bubble(best_roll, sub_t[-1])

        rolling_rows.append({
            "window_start": sub[DATE_COL].iloc[0],
            "window_end": sub[DATE_COL].iloc[-1],
            "tc_index": best_roll["tc"],
            "tc_days_ahead": tc_days_ahead,
            "tc_date_approx": tc_date_approx,
            "m": best_roll["m"],
            "w": best_roll["w"],
            "A": best_roll["A"],
            "B": best_roll["B"],
            "C1": best_roll["C1"],
            "C2": best_roll["C2"],
            "r2": best_roll["r2"],
            "bubble_flag": bubble_flag,
        })

    rolling_df = pd.DataFrame(rolling_rows)
    rolling_df.to_csv(os.path.join(OUT_DIR, f"lppls_rolling_{target_col}.csv"), index=False, encoding="utf-8-sig")

    # Detections only
    detected_df = rolling_df[rolling_df["bubble_flag"] == 1].copy()
    detected_df.to_csv(os.path.join(OUT_DIR, f"lppls_detected_{target_col}.csv"), index=False, encoding="utf-8-sig")

    # Bubble confidence = rolling mean of flags
    if len(rolling_df) > 0:
        plot_df = rolling_df.copy()
        plot_df["window_end"] = pd.to_datetime(plot_df["window_end"])
        plot_df = plot_df.sort_values("window_end").reset_index(drop=True)
        plot_df["bubble_confidence"] = plot_df["bubble_flag"].rolling(5, min_periods=1).mean()

        plt.figure(figsize=(12, 5))
        plt.plot(plot_df["window_end"], plot_df["bubble_confidence"])
        plt.title(f"LPPLS Bubble Confidence - {target_col}")
        plt.xlabel("Window End")
        plt.ylabel("Bubble Confidence (rolling mean of flags)")
        plt.ylim(-0.05, 1.05)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f"lppls_confidence_{target_col}.png"), dpi=300)
        plt.close()

    # Console summary
    print("=" * 70)
    print(f"[TARGET] {target_col}")
    print(f"Full-sample R^2          : {best_full['r2']:.4f}")
    print(f"Full-sample tc (index)   : {best_full['tc']:.3f}")
    print(f"Full-sample tc approx    : {summary_df.loc[0, 'tc_date_approx']}")
    print(f"m                        : {best_full['m']:.4f}")
    print(f"w                        : {best_full['w']:.4f}")
    print(f"B                        : {best_full['B']:.6f}")
    print(f"Positive bubble flag     : {summary_df.loc[0, 'is_positive_bubble']}")
    print(f"Rolling detections count : {len(detected_df)}")
    print(f"Saved to                 : {OUT_DIR}")


# =========================
# Main
# =========================
def main():
    df = pd.read_csv(INPUT_CSV)

    if DATE_COL not in df.columns:
        raise ValueError(f"DATE_COL '{DATE_COL}' not found in columns: {df.columns.tolist()}")

    for col in TARGET_COLS:
        if col not in df.columns:
            print(f"[SKIP] {col} not found.")
            continue
        fit_one_series(df, col)

    print("\nDone.")


if __name__ == "__main__":
    main()

[TARGET] COPPER
Full-sample R^2          : 0.6013
Full-sample tc (index)   : 1325.000
Full-sample tc approx    : 2026-01-13 00:00:00
m                        : 0.5121
w                        : 6.3383
B                        : -0.010466
Positive bubble flag     : 0
Rolling detections count : 86
Saved to                 : ./lppls_out
[TARGET] SOLVPN
Full-sample R^2          : 0.7719
Full-sample tc (index)   : 1431.354
Full-sample tc approx    : 2026-04-29 00:00:00
m                        : 0.6216
w                        : 6.0000
B                        : -0.007053
Positive bubble flag     : 1
Rolling detections count : 110
Saved to                 : ./lppls_out

Done.
